# Demo 04 — Recursive Language Models (RLM) with DSPy

The runnable notebook behind **§13 (Recursive Language Models)** of *Chapter 04 — Inference-Time
Retrieval Patterns*.

Every other demo in this chapter puts data in front of the LLM by **retrieving chunks** — RAG.
RLM is the other move: skip the index, load the whole document into a sandboxed Python REPL, and
let the LLM **explore it with code** — printing samples, searching, filtering — calling itself
recursively on interesting slices, until it has enough to answer.

This notebook runs `dspy.RLM` on a real 29KB synthetic annual report
(`workshops/workshop4/large_report.txt`, reused as-is): the exact §13.1 example, the exploration
trace itself (§13.3's "transparency" claim, made concrete), tools (§13.2), and an honest
measured comparison against just stuffing the document into a single prompt.

**Requires Deno** (RLM sandboxes Python execution via Pyodide/WASM):
`curl -fsSL https://deno.land/install.sh | sh` (macOS/Linux). Also `dspy>=3.2`, `anthropic`.
Needs an `ANTHROPIC_API_KEY` in a `.env` file — on Colab, `google.colab` (preinstalled) supplies
it from Colab Secrets (sidebar key icon) instead.

In [ ]:
import os
os.environ["PATH"] += os.pathsep + os.path.expanduser("~/.deno/bin")  # RLM needs `deno` on PATH

import subprocess
deno_check = subprocess.run(["deno", "--version"], capture_output=True, text=True)
assert deno_check.returncode == 0, (
    "Deno not found. Install it: curl -fsSL https://deno.land/install.sh | sh"
)
print(deno_check.stdout.splitlines()[0])

from dotenv import load_dotenv
load_dotenv()
# On Colab, pull the key from Colab Secrets (sidebar key icon) if it isn't already in the env.
if not os.environ.get("ANTHROPIC_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set — add it to your .env file"

import time
import dspy

# cache=False: this notebook measures latency/call-count honestly, so every call must be fresh.
lm = dspy.LM("anthropic/claude-haiku-4-5-20251001", cache=False)
dspy.configure(lm=lm)
print("ready")

## 1 — The document

A synthetic annual operations report for a fictional company (Nexara Technologies) — 29KB, ~600
lines: an executive summary, quarterly deep-dives (Q1–Q4), department sections, a risk register,
and financial tables. Small enough to fit in a single prompt today, which is exactly what lets us
run a fair, honest comparison later in this notebook.

In [2]:
with open("large_report.txt") as f:
    document = f.read()

print(f"{len(document):,} characters, {document.count(chr(10)) + 1} lines")
print(document[:400], "...")

29,080 characters, 596 lines
NEXARA TECHNOLOGIES — ANNUAL OPERATIONS & STRATEGY REPORT
Fiscal Year 2025
CONFIDENTIAL — Internal Distribution Only

TABLE OF CONTENTS
-----------------
1. Executive Summary
2. Company Overview
3. Q1 2025 — January through March
4. Q2 20 ...


## 2 — Baseline: just stuff it into one prompt

Before reaching for RLM, try the obvious thing: the whole document fits in context, so hand it to
the LLM directly in a single call.

In [3]:
stuff = dspy.Predict("document, question -> answer")

t0 = time.time()
stuffed_result = stuff(document=document, question="What were the key findings from Q3?")
stuffed_time = time.time() - t0

print(f"took {stuffed_time:.1f}s, {len(lm.history)} LM call\n")
print(stuffed_result.answer)

took 10.9s, 1 LM call

The key findings from Q3 2025 were:

1. **Nexara AI Suite Launch Exceeded All Targets** — The AI Suite launch on July 22 was the most successful product release in company history. Within 10 weeks: 412 existing customers upgraded generating $4.1M in expansion ARR, 68 net-new logos cited it as primary purchase driver, the Sentinel anomaly detection feature reduced mean time to alert by 73%, and product-led growth contributed 19 new Professional-tier customers. CSAT for AI features was 4.7/5.0.

2. **Record Net New ARR** — Q3 achieved $6.8M in new ARR (the highest single-quarter figure in company history), surpassing the previous record by 55%. This included $4.1M from AI Suite expansion, $2.4M from 61 new logos (avg. ACV $39,300), and $0.3M from Vanta Signal co-sell revenue, resulting in net new ARR of $6.3M.

3. **Vanta Signal Acquisition Completed Ahead of Schedule** — The $11.2M acquisition closed August 5 with 11-person team integration completed 11 days early

## 3 — RLM: explore instead of stuffing  (§13.1)

Same question, but the LLM never sees the full document in its prompt — only metadata (type,
length, a preview). It writes Python to explore `document` as a REPL variable, calling
`llm_query(...)` for semantic sub-analysis, until it calls `SUBMIT(...)`.

In [4]:
lm.history.clear()  # isolate the call count for this section

rlm = dspy.RLM("document, question -> answer", max_iterations=10)

t0 = time.time()
rlm_result = rlm(document=document, question="What were the key findings from Q3?")
rlm_time = time.time() - t0

print(f"took {rlm_time:.1f}s, {len(lm.history)} LM call(s) "
      f"({len(rlm_result.trajectory)} REPL iterations)\n")
print(rlm_result.answer)

took 47.2s, 7 LM call(s) (7 REPL iterations)

KEY FINDINGS FROM Q3 2025 (July through September):

1. NEXARA AI SUITE LAUNCH EXCEEDED ALL TARGETS
   - 412 existing customers upgraded to AI-enabled tiers, generating $4.1M in expansion ARR
   - 68 net-new logos cited the AI Suite as primary purchase driver
   - AI-assisted anomaly detection feature ("Sentinel") reduced mean time to alert by 73%
   - 19 new Professional-tier customers through product-led growth (new channel)
   - Customer satisfaction (CSAT): 4.7/5.0 (n=312 responses)

2. RECORD NET NEW ARR
   - Q3 new ARR: $6.8M (highest single-quarter in company history, 55% above previous record)
   - AI Suite expansion: $4.1M
   - New logos: 61 at average ACV of $39,300 (totaling $2.4M)
   - Co-sell revenue from Vanta Signal partners: $0.3M
   - Net new ARR: $6.3M (with churned ARR at $0.5M)

3. VANTA SIGNAL ACQUISITION COMPLETED AND INTEGRATED AHEAD OF SCHEDULE
   - Acquired for $11.2M on August 5, 2025
   - Streaming engine embedded

## 4 — The exploration is inspectable  (§13.3)

§13.3 claims RLM's transparency is "high — the exploration code is inspectable," against RAG's
"low — hard to know why a chunk was retrieved." `result.trajectory` is the receipt: every step's
reasoning, the Python it ran, and what the REPL printed back.

In [5]:
for i, step in enumerate(rlm_result.trajectory):
    print(f"=== step {i} ===")
    print("reasoning:", step["reasoning"][:200].replace(chr(10), " "), "...")
    print("code:")
    print("  " + step["code"].replace(chr(10), chr(10) + "  "))
    print("output:", step["output"][:150].replace(chr(10), " "), "...")
    print()

=== step 0 ===
reasoning: Let me analyze the task: 1. I need to find the key findings from Q3 in the provided document 2. The document is a Nexara Technologies Annual Operations & Strategy Report for FY2025 3. Q3 refers to "Q3 ...
code:
  # First, let's examine the document structure and find the Q3 section
  print("Document length:", len(document))
  print("\nFirst 1000 characters:")
  print(document[:1000])
  print("\n" + "="*80)
  print("\nSearching for Q3 section...")
  
  # Look for Q3 content
  q3_index = document.find("Q3 2025")
  if q3_index != -1:
      print(f"Found Q3 2025 at character position {q3_index}")
      # Extract a section around Q3
      q3_section = document[q3_index:q3_index+3000]
      print("\nQ3 Section (first 3000 chars):")
      print(q3_section)
  else:
      print("Q3 2025 not found, searching for variations...")
      # Try other variations
      for variant in ["Q3", "third quarter", "July", "September"]:
          if variant in document:
              

## 5 — Providing tools to RLM  (§13.2)

The REPL's sandbox has the standard library plus `llm_query` and `SUBMIT`. A custom tool is any
Python callable defined *outside* the sandbox — it runs in the host environment, so it can hit the
filesystem, an API, or (as here) just the system clock, and its return value comes back into the
REPL as a string.

In [6]:
def get_current_date() -> str:
    '''Returns today's date in ISO format.'''
    import datetime
    return datetime.date.today().isoformat()

lm.history.clear()
rlm_with_tools = dspy.RLM("document, question -> answer", tools=[get_current_date], max_iterations=10)

tool_result = rlm_with_tools(
    document=document,
    question="The report covers Q3 2025. How many months ago was that from today?",
)
print(tool_result.answer)

9 months


## 6 — RLM vs. stuffing: an honest, measured comparison

On a 29KB document, both approaches land on the same answer. The difference is cost and latency,
not correctness — and that difference is the whole point of §13.3's table. RLM's advantage isn't
"better answers on documents that already fit in context." It's the documents that **don't**: the
chapter cites RLMs running over 400MB of logs, which no single prompt could ever hold.

In [7]:
print(f"{'approach':20} {'time':>8} {'LM calls':>10}")
print(f"{'single-prompt stuff':20} {stuffed_time:>7.1f}s {1:>10}")
print(f"{'RLM (explore)':20} {rlm_time:>7.1f}s {len(rlm_result.trajectory):>10}")
print()
print("Same document, same question, same-quality answer — RLM paid ~"
      f"{rlm_time / stuffed_time:.0f}x the latency and {len(rlm_result.trajectory)}x the LM calls "
      "for nothing extra *at this scale*. That trade only pays off once the document can't fit in "
      "a prompt at all.")

approach                 time   LM calls
single-prompt stuff     10.9s          1
RLM (explore)           47.2s          7

Same document, same question, same-quality answer — RLM paid ~4x the latency and 7x the LM calls for nothing extra *at this scale*. That trade only pays off once the document can't fit in a prompt at all.


## Takeaways

- **RLM turns a context-window problem into a coding-and-reasoning problem.** No chunking,
  embedding, or index — just load the data as a REPL variable and let the LLM explore it.
- **On a document that already fits in context, stuffing wins on cost and latency.** RLM's
  measured overhead here (more LM calls, much slower) bought nothing extra — the value shows up
  only past the point where a single prompt can't hold the source at all.
- **`result.trajectory` makes the exploration inspectable** — a real transparency advantage over
  RAG, where you often can't tell *why* a chunk was retrieved.
- **Tools extend the REPL with anything outside the sandbox** — current date, live lookups,
  external APIs — passed to `dspy.RLM(..., tools=[...])` and callable by name from inside.
- **Use RAG** for a large, stable corpus with predictable queries and latency budgets. **Use RLM**
  for one very long, unstructured source and exploratory queries. **Hybrid**: RAG to shortlist,
  RLM to reason over what it found.